# 01 - Exploracao da base

Base escolhida: `cellphone.ibyte.json`.

Objetivo do notebook: entender o formato dos dados brutos, extrair os titulos dos
produtos, medir o volume disponivel para anotacao e levantar o vocabulario que
alimenta a pre-anotacao por regras (notebook 02).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import collections
import re

import pandas as pd

from src.dados import (
    carregar_produtos,
    titulos_unicos,
    marcas_do_catalogo,
    salvar_jsonl,
    carregar_jsonl,
    DIR_ANOT,
)

BASE = "cellphone"

## 1. Leitura dos dados brutos

Os arquivos em `data/raw/` nao sao listas de titulos: sao capturas de microdata
do site, em que cada registro traz um objeto `Product` (nome, marca, sku) e um
`BreadcrumbList` com a categoria. A extracao esta em `src/dados.py`.

In [ ]:
produtos = carregar_produtos(BASE)
print(f"registros com produto: {len(produtos)}")

df = pd.DataFrame(produtos)
df.head(10)

## 2. Deduplicacao

O mesmo produto aparece varias vezes (paginacao e vitrines repetidas). A anotacao
e feita sobre titulos unicos.

In [ ]:
unicos = titulos_unicos(produtos)
print(f"titulos unicos: {len(unicos)}  ({len(unicos) / len(produtos):.1%} do total)")

df_unicos = pd.DataFrame(unicos)
df_unicos["n_caracteres"] = df_unicos["titulo"].str.len()
df_unicos["n_tokens"] = df_unicos["titulo"].str.split().str.len()
df_unicos[["n_caracteres", "n_tokens"]].describe()

## 3. Distribuicao por categoria

A categoria vem do breadcrumb e mostra que a base mistura o produto principal
(smartphones) com acessorios (capas, peliculas, carregadores). Isso influencia a
definicao das tags: em um acessorio, o modelo citado no titulo e compatibilidade,
nao o produto.

In [ ]:
contagem = collections.Counter(p["categoria"] for p in unicos)
pd.Series(dict(contagem.most_common())).to_frame("produtos")

## 4. Marcas declaradas no catalogo

O campo `brand` do microdata da um dicionario de marcas confiavel, usado como
gazetteer na pre-anotacao.

In [ ]:
marcas = marcas_do_catalogo(unicos)
print(f"marcas distintas: {len(marcas)}")
print(marcas)

## 5. Ruido conhecido da base

Parte dos titulos vem truncada no caractere de aspas usado para polegadas
(`Tela de 6,5` sem o fechamento). E uma limitacao da captura original e precisa
ser considerada na leitura dos resultados: alguns titulos perdem atributos que
apareceriam no final.

In [ ]:
truncados = [p["titulo"] for p in unicos if re.search(r"\d[.,]\d$|\bTela( de)? \d", p["titulo"])]
print(f"titulos com indicio de truncamento: {len(truncados)}")
for titulo in truncados[:10]:
    print(" -", titulo)

## 6. Vocabulario mais frequente

Os tokens mais comuns orientam quais tags valem a pena definir.

In [ ]:
tokens = collections.Counter()
for produto in unicos:
    tokens.update(t.lower().strip(",.-") for t in produto["titulo"].split())

pd.Series(dict(tokens.most_common(40))).to_frame("frequencia")

## Conclusoes

- 1032 titulos unicos disponiveis para anotacao.
- A base mistura produto principal e acessorio, o que exige uma regra explicita
  no guia de anotacao para o caso `Capa para iPhone 13`.
- O campo `brand` cobre 60 marcas e pode ser reaproveitado como dicionario.
- Atributos recorrentes nos titulos: armazenamento, memoria RAM, cor, tamanho de
  tela, geracao de rede, camera e codigo do fabricante.